In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, regexp_extract, lit,
    to_timestamp, month, dayofmonth, year,
    concat_ws
)
from pyspark.sql.types import StringType
import time

# ─────────────────────────────────────────────
# TIMER
# ─────────────────────────────────────────────
start = time.time()
print(f"[{start:.3f}][SPARK-JOB] Iniciando execução do Job...")

# ─────────────────────────────────────────────
# PACOTES CLICKHOUSE
# ─────────────────────────────────────────────
packages = ",".join([
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.10.0",
    "com.clickhouse:clickhouse-client:0.7.0",
    "com.clickhouse:clickhouse-http-client:0.7.0",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1",
])

# ─────────────────────────────────────────────
# SESSÃO SPARK
# ─────────────────────────────────────────────
spark = (
    SparkSession.builder
    .config("spark.jars.packages", packages)

    # Paralelismo
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")

    # Adaptive Query Execution
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")

    # Joins
    .config("spark.sql.autoBroadcastJoinThreshold", "10mb")

    # Compressão de shuffle
    .config("spark.shuffle.compress", "true")
    .config("spark.io.compression.codec", "lz4")

    # Serialização
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")

    # Suprime o WARN do toYYYYMM — reparticionamento feito manualmente abaixo
    .config("spark.clickhouse.ignoreUnsupportedTransform", "true")

    .getOrCreate()
)

# ─────────────────────────────────────────────
# CLICKHOUSE — catalog
# ─────────────────────────────────────────────
ch_conf = {
    "spark.sql.catalog.clickhouse":             "com.clickhouse.spark.ClickHouseCatalog",
    "spark.sql.catalog.clickhouse.host":        "clickhouse",
    "spark.sql.catalog.clickhouse.protocol":    "http",
    "spark.sql.catalog.clickhouse.http_port":   "8123",
    "spark.sql.catalog.clickhouse.user":        "admin",
    "spark.sql.catalog.clickhouse.password":    "admin",
    "spark.sql.catalog.clickhouse.database":    "logs",
    "spark.clickhouse.write.format":            "json",
    "spark.clickhouse.write.batchSize":         "100000",
    "spark.clickhouse.write.maxRetries":        "3",
}
for k, v in ch_conf.items():
    spark.conf.set(k, v)

# ─────────────────────────────────────────────
# REGEX — formato real: Jan/02/2026 15:14:31 routing,info: mensagem
# ─────────────────────────────────────────────
LOG_PATTERN = r"^(\w{3}/\d{2}/\d{4})\s(\d{2}:\d{2}:\d{2})\s([^,]+),([^:]+):\s(.*)$"

# ─────────────────────────────────────────────
# LEITURA
# ─────────────────────────────────────────────
df = spark.read.text("/opt/spark-data/logs.log")
df.show(truncate=False)

pattern = r"^(\w{3}/\d{2}/\d{4}) (\d{2}:\d{2}:\d{2}) ([\w\-]+),(\w+): (.*)$"
parsed = df.select(
    regexp_extract("value", pattern, 1).alias("date"),
    regexp_extract("value", pattern, 2).alias("time"),
    regexp_extract("value", pattern, 3).alias("service"),
    regexp_extract("value", pattern, 4).alias("level"),
    regexp_extract("value", pattern, 5).alias("message")
)

structured = parsed \
    .withColumn("timestamp_str", concat_ws(" ", col("date"), col("time"))) \
    .withColumn("timestamp", to_timestamp("timestamp_str", "MMM/dd/yyyy HH:mm:ss")) \
    .withColumn("month", month("timestamp")) \
    .withColumn("day", dayofmonth("timestamp")) \
    .withColumn("year", year("timestamp")) \
    .select(
        "timestamp",
        "month",
        "day",
        "year",
        "time",
        "service",
        "level",
        "message"
    )





# ─────────────────────────────────────────────
# ESCRITA → ClickHouse
# ─────────────────────────────────────────────
(
    structured
    .writeTo("clickhouse.logs.router_logs")
    .append()
)

# ─────────────────────────────────────────────
# FIM
# ─────────────────────────────────────────────
spark.stop()
end = time.time()
print(f"[{end:.3f}][SPARK-JOB] Job finalizado com sucesso.")
print(f"Job executado em: {end - start:.2f} segundos")

[1777295811.328][SPARK-JOB] Iniciando execução do Job...
+---------------------------------------------------------------------------------------------------------+
|value                                                                                                    |
+---------------------------------------------------------------------------------------------------------+
|Jan/02/2026 15:14:31 routing,info: firewall accepted packet from 105.89.221.80                           |
|Jan/08/2026 14:12:23 interface,link: wireless client connected mac 24:92:95:31:6F:14 ip 187.23.191.189   |
|Jan/08/2026 02:56:37 dhcp,info: user admin logged in via winbox                                          |
|Feb/02/2026 05:59:14 system,info: BGP peer established                                                   |
|Jan/28/2026 16:01:06 wireless,warning: wireless client disconnected mac 96:E1:C1:1E:C8:37 ip 79.230.14.58|
|Jan/08/2026 10:11:59 interface,link: DHCP lease expired ip 112.185.228.71 mac 

26/04/27 13:16:52 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
26/04/27 13:16:52 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
                                                                                

[1777295818.415][SPARK-JOB] Job finalizado com sucesso.
Job executado em: 7.09 segundos
